In [1]:
!pip install newspaper3k nltk sumy transformers torch scikit-learn

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from newspaper import Article
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from collections import Counter
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

from transformers import pipeline
!pip install -U transformers sentencepiece

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
url = "https://www.usatoday.com/story/news/politics/2025/06/13/pete-hegseth-pentagon-invade-greenland-plan/84188458007/"

article = Article(url)
article.download()
article.parse()

text = article.text

print(text[:1000])  # preview

June 13, 2025, 2:33 p.m. ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.

Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."

"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct? Because I sure as hell hope that it is not your testimony," Turner dug in.

"We look forward to working with Greenland to ensure that it is secured from any potential threats," Hegseth said.

President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary. He has insisted that acquiring Greenland is necessary for national security, citing growing Chinese and Russian influence in the region. The island is also rich in cri

# 1. Extract and print an extractive summary using the TextRank method.

In [3]:
parser = PlaintextParser.from_string(text, Tokenizer("english"))
summarizer = TextRankSummarizer()

summary = summarizer(parser.document, 5)

print("TextRank Summary:\n")
for sentence in summary:
    print(sentence)

TextRank Summary:

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.
Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."
The island is also rich in critical minerals that the U.S. wants to challenge Chinese monopolies in some industries, USA TODAY has reported.
During a March visit to Pituffik Space Base, the U.S. base on Greenland, Vice President JD Vance accused Denmark of "failing" to protect the Arctic island while downplaying Trump's threats to take it over by force.
In the latest snub to Denmark and other European allies, the Pentagon reportedly plans to move its oversight of the island from U.S. European Command to U.S. Northern Command.


# 2. Extract and print an extractive summary using a frequency-based sentence scoring method.

In [4]:
from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(text)

In [5]:
stop_words = set(stopwords.words("english"))

words = nltk.word_tokenize(text.lower())
words = [w for w in words if w.isalnum()]

word_freq = Counter(words)

for word in list(word_freq.keys()):
    if word in stop_words:
        del word_freq[word]

sentence_scores = {}

for sentence in sentences:
    for word in nltk.word_tokenize(sentence.lower()):
        if word in word_freq:
            if sentence not in sentence_scores:
                sentence_scores[sentence] = word_freq[word]
            else:
                sentence_scores[sentence] += word_freq[word]

# Top 5 sentences
summary_freq = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:5]

print("Frequency-based Summary:\n")
for s in summary_freq:
    print(s)

Frequency-based Summary:

Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."
During a March visit to Pituffik Space Base, the U.S. base on Greenland, Vice President JD Vance accused Denmark of "failing" to protect the Arctic island while downplaying Trump's threats to take it over by force.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.
President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary.
In the latest snub to Denmark and other European allies, the Pentagon reportedly plans to move its oversight of the island from U.S. European Command to U.S. Northern Command.


# 3. Generate and print an abstractive summary using a pretrained transformer model (e.g., BART or T5).

In [8]:
!pip install -U transformers sentencepiece torch

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(text[:2000], return_tensors="pt", truncation=True, max_length=1024)
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=130,
    min_length=40,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Abstractive Summary (BART):\n")
print(summary)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Abstractive Summary (BART):

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary.


# 4. Print a Lead-3 summary (the first three sentences of the article).

In [10]:
lead3 = sentences[:3]

print("Lead-3 Summary:\n")
for s in lead3:
    print(s)

Lead-3 Summary:

June 13, 2025, 2:33 p.m.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.
Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."


# 5. Print a manual compression summary, limiting the result to about 20% of the original sentences.

In [11]:
num_sentences = int(len(sentences) * 0.2)

compressed_summary = sentences[:num_sentences]

print("Manual Compression Summary (20%):\n")
for s in compressed_summary:
    print(s)

Manual Compression Summary (20%):

June 13, 2025, 2:33 p.m.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.


# 6. Use an LLM to summary the text.

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = f"""
Please provide a concise summary of the following news article:

{text[:2000]}
"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

output_ids = model.generate(
    inputs["input_ids"],
    max_length=150,
    min_length=40,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("LLM Summary:\n")
print(summary)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM Summary:

"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct? Because I sure as hell hope that it is not your testimony," Rep. Mike Turner said.
